 Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc

# Load the cleaned regression dataset
file_path = "Input_Data/01_cleaned_data/uwb_dataset_regression.parquet"
df = pd.read_parquet(file_path)

# Separate features (X) and target (y)
target_col = "TARGET_Second_Path_Range"
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found.")

features = [col for col in df.columns if col not in [target_col, 'NLOS']]
X = df[features]
y = df[target_col]

# Split into 80% Training and 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dataset loaded. Training on {X_train.shape[0]} rows and {X_train.shape[1]} features.")

# Store each model's results for the final comparison
model_results = []

# Plot model performance and store metrics
def plot_independent_dashboard(model_name, grid, param_name, train_time_sec=None):
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    cv_res = pd.DataFrame(grid.cv_results_)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    model_results.append({
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Train Time (s)': train_time_sec,
        'Best Params': grid.best_params_
    })

    residuals = y_test - y_pred

    fig, axes = plt.subplots(4, 1, figsize=(8, 20))
    fig.suptitle(f"Model Results: {model_name} (Regression)", fontsize=18, fontweight='bold', y=0.98)

    # Graph 1: Train vs CV RMSE (learning curve)
    param_values = [str(x) for x in cv_res[f'param_{param_name}']]
    train_rmse = -cv_res['mean_train_score']
    test_rmse = -cv_res['mean_test_score']
    axes[0].plot(param_values, train_rmse, marker='s', label='Train RMSE', color='skyblue', lw=2)
    axes[0].plot(param_values, test_rmse, marker='o', label='CV RMSE', color='salmon', lw=2)
    axes[0].set_title("Learning Curve: Train vs. CV RMSE")
    axes[0].set_xlabel(param_name)
    axes[0].set_ylabel("RMSE (lower is better)")
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=30)

    # Graph 2: Predicted vs Actual (fit quality)
    axes[1].scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    axes[1].set_title("Predicted vs Actual")
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predicted")
    axes[1].legend()

    # Graph 3: Residual Distribution (error spread)
    sns.histplot(residuals, bins=40, kde=True, ax=axes[2], color='darkorange')
    axes[2].axvline(0, color='black', linestyle='--', linewidth=1.2)
    axes[2].set_title("Residual Distribution")
    axes[2].set_xlabel("Residual (Actual - Predicted)")

    # Graph 4: Residuals vs Predicted (scatter)
    axes[3].scatter(y_pred, residuals, alpha=0.3, s=10, color='purple')
    axes[3].axhline(0, color='black', linestyle='--', linewidth=1.2)
    axes[3].set_title("Residuals vs Predicted")
    axes[3].set_xlabel("Predicted")
    axes[3].set_ylabel("Residual")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    print(f"Best Params: {grid.best_params_}")
    print(f"Test RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")
    if train_time_sec is not None:
        print(f"Training Time: {train_time_sec:.2f} seconds")

Data Cleaning & Preprocessing

In [ ]:
# Drop duplicates and handle nulls
df.dropna(inplace=True)

# Separate CIR columns (1016 samples) from scalar features
cir_cols = [c for c in df.columns if 'cir' in c.lower() and c != 'CIR_PWR']
scalar_cols = ['RANGE', 'FP_IDX', 'FP_AMP1', 'FP_AMP2', 'FP_AMP3',
               'STDEV_NOISE', 'CIR_PWR', 'MAX_NOISE', 'RXPACC', 'FRAME_LEN', 'PREAM_LEN']

X_scalar = df[scalar_cols]
X_cir    = df[cir_cols]        # shape: (N, 1016)
y        = df['TARGET_Second_Path_Range']         # regression target

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

# CIR features
scaler_cir = StandardScaler()
X_cir_scaled = scaler_cir.fit_transform(X_cir)

# # Fit PCA on full CIR data (all 1016 components)
# pca_full = PCA(random_state=42)
# pca_full.fit(X_cir_scaled)

# cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

# # Find component counts for common thresholds
# for thresh in [80, 90, 95, 99]:
#     n = np.argmax(cumvar >= thresh) + 1
#     print(f"{thresh}% variance retained → {n} components")

In [ ]:
import pandas as pd
import numpy as np

# Make a copy of the original dataframe to keep it safe
df_clean = df.copy()
print(f"Original Dataset Shape: {df_clean.shape}\n")

# ==========================================
# 1. OUTLIER REMOVAL (Using IQR Method)
# ==========================================
print("--- 1. Outlier Removal ---")
# We will check the hardware noise columns. You can add 'CIR_PWR' or 'RXPACC' to this list if needed.
columns_to_check = ['STDEV_NOISE', 'MAX_NOISE'] 

for col in columns_to_check:
    if col in df_clean.columns:
        # Calculate Q1 (25th percentile) and Q3 (75th percentile)
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        
        # Calculate Interquartile Range (IQR)
        IQR = Q3 - Q1

        # Define bounds for what is considered an "outlier" (1.5x IQR is the statistical standard)
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Count how many outliers exist before removing them
        outliers_count = len(df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)])
        print(f"Found {outliers_count} extreme outliers in '{col}'.")

        # Filter the dataframe to KEEP only rows within the normal bounds
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

print(f"Dataset Shape after Outlier Removal: {df_clean.shape}\n")


# ==========================================
# 2. DATA LEAKAGE CHECK (Using Correlation)
# ==========================================
print("--- 2. Data Leakage Check ---")
# Data leakage usually happens if a feature is basically a direct mathematical proxy for your target.
# We test this by checking the Pearson Correlation.

# To keep the calculation fast and readable, we will exclude the 1000+ raw CIR columns for this check
non_cir_cols = [c for c in df_clean.columns if not c.startswith('CIR') or c == 'CIR_PWR']

# Calculate correlation matrix for the non-CIR columns
correlation_matrix = df_clean[non_cir_cols].corr()

# Check correlations against the regression target: RANGE
if 'TARGET_Second_Path_Range' in correlation_matrix.columns:
    # Get the absolute correlation values, sort them highest to lowest
    range_corr = correlation_matrix['TARGET_Second_Path_Range'].abs().sort_values(ascending=False)
    
    print("Top features most correlated with the target 'TARGET_Second_Path_Range':")
    # We skip the 1st one (.head(6)[1:]) because RANGE always has a 1.0 correlation with itself
    print(range_corr.head(6)[1:]) 
    
    # Check for dangerously high correlations (Greater than 95%)
    potential_leaks = range_corr[(range_corr > 0.95) & (range_corr < 1.0)]
    
    if not potential_leaks.empty:
        print("\n⚠️ WARNING: Potential Data Leakage Detected!")
        print("These features have > 0.95 correlation with RANGE. If they are distance estimates rather than raw hardware metrics, you must drop them before training:")
        print(potential_leaks)
    else:
        print("\n✅ Good news: No obvious data leakage detected (No features correlate > 0.95 with RANGE).")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor

# 1. Exclude second path target and NLOS; keep RANGE as a feature
targets = ['NLOS', 'TARGET_Second_Path_Range']
cir_cols = [c for c in df_clean.columns if c.startswith('CIR') and c != 'CIR_PWR']
standard_cols = [c for c in df_clean.columns if c not in targets and c not in cir_cols]

# 2. X includes RANGE now; y is the second path distance
X = df_clean[standard_cols + cir_cols]
y = df_clean['TARGET_Second_Path_Range']

# 3. No stratify for regression
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4–5. Preprocessor stays exactly the same
cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA())
])
standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])

# 6. Regressor instead of classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))
])

# 7. RMSE scoring instead of accuracy
param_grid = {
    'preprocessor__cir_processing__pca__n_components': [10, 15, 20, 25, 30, 35, 40, 50]
}

print("Starting Grid Search for optimal PCA components (Q2 Distance Estimator)...")
grid_search = GridSearchCV(pipeline, param_grid, cv=5,
                           scoring='neg_root_mean_squared_error', verbose=1)
grid_search.fit(X_train, y_train)

print("\n" + "="*40)
print(f"✅ BEST PCA COMPONENTS : {grid_search.best_params_['preprocessor__cir_processing__pca__n_components']}")
print(f"✅ BEST CROSS-VAL RMSE : {-grid_search.best_score_:.4f} m")
print("="*40 + "\n")

results = pd.DataFrame(grid_search.cv_results_)
print("RMSE for each number of components tested:")
for index, row in results.iterrows():
    comps = row['params']['preprocessor__cir_processing__pca__n_components']
    rmse  = -row['mean_test_score']
    print(f" - {comps:2d} components: RMSE = {rmse:.4f} m")

PCA 

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

# Import Classifiers
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# ==========================================
# 1. LOCK IN THE OPTIMAL PREPROCESSOR
# ==========================================
OPTIMAL_PCA_COMPONENTS = grid_search.best_params_['preprocessor__cir_processing__pca__n_components']


print(f"Building preprocessor with {OPTIMAL_PCA_COMPONENTS} CIR PCA components...\n")

cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=OPTIMAL_PCA_COMPONENTS)) 
])

standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# (Assumes cir_cols and standard_cols are still stored in memory from your previous code)
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])



Train/Test Split

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import Regressors
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR

# OPTIMAL_PCA_COMPONENTS = grid_search.best_params_['preprocessor__cir_processing__pca__n_components']

models_and_params = {
    "Random Forest": {
        "model": RandomForestRegressor(random_state=42, n_jobs=-1),
        "params": {
            'classifier__n_estimators': [50, 100],
            'classifier__max_depth': [10, 20, None]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            'classifier__max_depth': [5, 10, 15, None],
            'classifier__min_samples_split': [2, 5, 10]
        }
    },
    "KNN": {
        "model": KNeighborsRegressor(),
        "params": {
            'classifier__n_neighbors': [3, 5, 7, 11],
            'classifier__weights': ['uniform', 'distance']
        }
    },
    "Ridge Regression": {
        "model": Ridge(),
        "params": {
            'classifier__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
        }
    },
    "Gradient Boosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "params": {
            'classifier__n_estimators': [50, 100],
            'classifier__learning_rate': [0.05, 0.1],
            'classifier__max_depth': [3, 5]
        }
    },
    "SVR (RBF Kernel)": {
        "model": SVR(kernel='rbf'),          # no probability=True for regression
        "params": {
            'classifier__C': [0.1, 1.0, 10.0],
            'classifier__gamma': ['scale', 'auto']
        }
    }
}

train_rmses = []
test_rmses  = []
model_names = []
results_dict = {}
best_estimators = {}


print("Tuning, Training, and Evaluating models. This may take a few minutes...\n")

In [ ]:
# ==========================================
# STEP 3: PREPROCESSING PIPELINE
# ==========================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Lock in the optimal number discovered in the previous grid search
# OPTIMAL_PCA_COMPONENTS = grid_search.best_params_['preprocessor__cir_processing__pca__n_components']
# del grid_search_rf
# gc.collect()

print(f"Initializing preprocessor with {OPTIMAL_PCA_COMPONENTS} PCA components...")

# Pipeline for raw CIR Waveform data
cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=OPTIMAL_PCA_COMPONENTS)) 
])

# Pipeline for standard hardware metrics
standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Combine them: This ensures the right math is applied to the right columns
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])

print("✅ Preprocessing pipeline ready.")

Random Forest

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Tuning and Evaluating: Random Forest...")
model_name_rf = "Random Forest"

# 1. Setup Model & Pipeline
rf_model  = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_params = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth'   : [10, 20, None]
}
pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', rf_model)])

# 2. Grid Search — RMSE instead of accuracy
grid_search_rf = GridSearchCV(pipeline_rf, rf_params, cv=3,
                               scoring='neg_root_mean_squared_error',
                               n_jobs=1, return_train_score=True)
grid_search_rf.fit(X_train, y_train)

# 3. Predict using Best Model
best_model_rf = grid_search_rf.best_estimator_
y_pred_rf     = best_model_rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
r2_rf   = r2_score(y_test, y_pred_rf)

# ==========================================
# GRAPHING: 1x3 Dashboard
# ==========================================
fig_rf, axes_rf = plt.subplots(3, 1, figsize=(8, 15))
fig_rf.suptitle(f'Comprehensive Evaluation: {model_name_rf}', fontsize=16, fontweight='bold')

# ── Graph 1: Hyperparameter Tuning Results ────────────────────────────────────
cv_results_rf  = pd.DataFrame(grid_search_rf.cv_results_)
param_labels_rf = [str(p).replace('classifier__', '') for p in cv_results_rf['params']]

# Negate scores (sklearn returns negative RMSE)
train_rmse_cv = -cv_results_rf['mean_train_score']
test_rmse_cv  = -cv_results_rf['mean_test_score']

axes_rf[0].plot(param_labels_rf, train_rmse_cv, label='Train RMSE', marker='s', ls='--', color='skyblue')
axes_rf[0].plot(param_labels_rf, test_rmse_cv,  label='CV RMSE',    marker='o', ls='-',  color='salmon')
axes_rf[0].set_title(f'{model_name_rf}: Tuning Results')
axes_rf[0].set_ylabel('RMSE (m)')
axes_rf[0].tick_params(axis='x', rotation=45)
axes_rf[0].legend()
axes_rf[0].grid(True, alpha=0.3)

# ── Graph 2: Predicted vs Actual ──────────────────────────────────────────────
axes_rf[1].scatter(y_test, y_pred_rf, alpha=0.2, s=5, color='steelblue')
lim = [min(y_test.min(), y_pred_rf.min()), max(y_test.max(), y_pred_rf.max())]
axes_rf[1].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
axes_rf[1].set_xlabel('Actual Range (m)')
axes_rf[1].set_ylabel('Predicted Range (m)')
axes_rf[1].set_title(f'{model_name_rf}: Predicted vs Actual\nRMSE={rmse_rf:.4f} m  R²={r2_rf:.4f}')
axes_rf[1].legend()
axes_rf[1].grid(alpha=0.3)

# ── Graph 3: Residuals Plot ───────────────────────────────────────────────────
residuals_rf = y_test.values - y_pred_rf
axes_rf[2].scatter(y_pred_rf, residuals_rf, alpha=0.2, s=5, color='darkorange')
axes_rf[2].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes_rf[2].set_xlabel('Predicted Range (m)')
axes_rf[2].set_ylabel('Residual (m)')
axes_rf[2].set_title(f'{model_name_rf}: Residuals\nMAE={mae_rf:.4f} m')
axes_rf[2].grid(alpha=0.3)

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

print(f"Best Params : {grid_search_rf.best_params_}")
print(f"RMSE        : {rmse_rf:.4f} m")
print(f"MAE         : {mae_rf:.4f} m")
print(f"R²          : {r2_rf:.4f}")


Decision

In [ ]:
print("Tuning and Evaluating: Decision Tree...")
model_name_dt = "Decision Tree"

dt_model  = DecisionTreeRegressor(random_state=42)
dt_params = {
    'classifier__max_depth'        : [5, 10, 15, None],
    'classifier__min_samples_split': [2, 5, 10]
}
pipeline_dt    = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', dt_model)])
grid_search_dt = GridSearchCV(pipeline_dt, dt_params, cv=3,
                               scoring='neg_root_mean_squared_error',
                               n_jobs=-1, return_train_score=True)
grid_search_dt.fit(X_train, y_train)

best_model_dt = grid_search_dt.best_estimator_
y_pred_dt     = best_model_dt.predict(X_test)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
mae_dt  = mean_absolute_error(y_test, y_pred_dt)
r2_dt   = r2_score(y_test, y_pred_dt)

fig_dt, axes_dt = plt.subplots(3, 1, figsize=(8, 15))
fig_dt.suptitle(f'Comprehensive Evaluation: {model_name_dt}', fontsize=16, fontweight='bold')

cv_results_dt   = pd.DataFrame(grid_search_dt.cv_results_)
param_labels_dt = [str(p).replace('classifier__', '') for p in cv_results_dt['params']]
axes_dt[0].plot(param_labels_dt, -cv_results_dt['mean_train_score'], label='Train RMSE', marker='s', ls='--', color='skyblue')
axes_dt[0].plot(param_labels_dt, -cv_results_dt['mean_test_score'],  label='CV RMSE',    marker='o', ls='-',  color='salmon')
axes_dt[0].set_title(f'{model_name_dt}: Tuning Results')
axes_dt[0].set_ylabel('RMSE (m)')
axes_dt[0].tick_params(axis='x', rotation=45)
axes_dt[0].legend(); axes_dt[0].grid(True, alpha=0.3)

axes_dt[1].scatter(y_test, y_pred_dt, alpha=0.2, s=5, color='steelblue')
lim = [min(y_test.min(), y_pred_dt.min()), max(y_test.max(), y_pred_dt.max())]
axes_dt[1].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
axes_dt[1].set_xlabel('Actual Range (m)'); axes_dt[1].set_ylabel('Predicted Range (m)')
axes_dt[1].set_title(f'{model_name_dt}: Predicted vs Actual\nRMSE={rmse_dt:.4f} m  R²={r2_dt:.4f}')
axes_dt[1].legend(); axes_dt[1].grid(alpha=0.3)

residuals_dt = y_test.values - y_pred_dt
axes_dt[2].scatter(y_pred_dt, residuals_dt, alpha=0.2, s=5, color='darkorange')
axes_dt[2].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes_dt[2].set_xlabel('Predicted Range (m)'); axes_dt[2].set_ylabel('Residual (m)')
axes_dt[2].set_title(f'{model_name_dt}: Residuals\nMAE={mae_dt:.4f} m')
axes_dt[2].grid(alpha=0.3)

plt.tight_layout(); plt.subplots_adjust(top=0.85); plt.show()
print(f"Best Params : {grid_search_dt.best_params_}")
print(f"RMSE={rmse_dt:.4f} m   MAE={mae_dt:.4f} m   R²={r2_dt:.4f}")


KNN

In [ ]:
print("Tuning and Evaluating: KNN...")
model_name_knn = "KNN"

knn_model  = KNeighborsRegressor()
knn_params = {
    'classifier__n_neighbors': [3, 5, 7, 11],
    'classifier__weights'    : ['uniform', 'distance']
}
pipeline_knn    = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', knn_model)])
grid_search_knn = GridSearchCV(pipeline_knn, knn_params, cv=3,
                                scoring='neg_root_mean_squared_error',
                                n_jobs=-1, return_train_score=True)
grid_search_knn.fit(X_train, y_train)

best_model_knn = grid_search_knn.best_estimator_
y_pred_knn     = best_model_knn.predict(X_test)
rmse_knn = np.sqrt(mean_squared_error(y_test, y_pred_knn))
mae_knn  = mean_absolute_error(y_test, y_pred_knn)
r2_knn   = r2_score(y_test, y_pred_knn)

fig_knn, axes_knn = plt.subplots(3, 1, figsize=(8, 15))
fig_knn.suptitle(f'Comprehensive Evaluation: {model_name_knn}', fontsize=16, fontweight='bold')

cv_results_knn   = pd.DataFrame(grid_search_knn.cv_results_)
param_labels_knn = [str(p).replace('classifier__', '') for p in cv_results_knn['params']]
axes_knn[0].plot(param_labels_knn, -cv_results_knn['mean_train_score'], label='Train RMSE', marker='s', ls='--', color='skyblue')
axes_knn[0].plot(param_labels_knn, -cv_results_knn['mean_test_score'],  label='CV RMSE',    marker='o', ls='-',  color='salmon')
axes_knn[0].set_title(f'{model_name_knn}: Tuning Results')
axes_knn[0].set_ylabel('RMSE (m)')
axes_knn[0].tick_params(axis='x', rotation=45)
axes_knn[0].legend(); axes_knn[0].grid(True, alpha=0.3)

axes_knn[1].scatter(y_test, y_pred_knn, alpha=0.2, s=5, color='steelblue')
lim = [min(y_test.min(), y_pred_knn.min()), max(y_test.max(), y_pred_knn.max())]
axes_knn[1].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
axes_knn[1].set_xlabel('Actual Range (m)'); axes_knn[1].set_ylabel('Predicted Range (m)')
axes_knn[1].set_title(f'{model_name_knn}: Predicted vs Actual\nRMSE={rmse_knn:.4f} m  R²={r2_knn:.4f}')
axes_knn[1].legend(); axes_knn[1].grid(alpha=0.3)

residuals_knn = y_test.values - y_pred_knn
axes_knn[2].scatter(y_pred_knn, residuals_knn, alpha=0.2, s=5, color='darkorange')
axes_knn[2].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes_knn[2].set_xlabel('Predicted Range (m)'); axes_knn[2].set_ylabel('Residual (m)')
axes_knn[2].set_title(f'{model_name_knn}: Residuals\nMAE={mae_knn:.4f} m')
axes_knn[2].grid(alpha=0.3)

plt.tight_layout(); plt.subplots_adjust(top=0.85); plt.show()
print(f"Best Params : {grid_search_knn.best_params_}")
print(f"RMSE={rmse_knn:.4f} m   MAE={mae_knn:.4f} m   R²={r2_knn:.4f}")


Ridge

In [ ]:
print("Tuning and Evaluating: Ridge Regression...")
model_name_ridge = "Ridge Regression"

ridge_model  = Ridge()
ridge_params = {
    'classifier__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}
pipeline_ridge    = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', ridge_model)])
grid_search_ridge = GridSearchCV(pipeline_ridge, ridge_params, cv=3,
                                  scoring='neg_root_mean_squared_error',
                                  n_jobs=-1, return_train_score=True)
grid_search_ridge.fit(X_train, y_train)

best_model_ridge = grid_search_ridge.best_estimator_
y_pred_ridge     = best_model_ridge.predict(X_test)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge  = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge   = r2_score(y_test, y_pred_ridge)

fig_ridge, axes_ridge = plt.subplots(3, 1, figsize=(8, 15))
fig_ridge.suptitle(f'Comprehensive Evaluation: {model_name_ridge}', fontsize=16, fontweight='bold')

cv_results_ridge   = pd.DataFrame(grid_search_ridge.cv_results_)
param_labels_ridge = [str(p).replace('classifier__', '') for p in cv_results_ridge['params']]
axes_ridge[0].plot(param_labels_ridge, -cv_results_ridge['mean_train_score'], label='Train RMSE', marker='s', ls='--', color='skyblue')
axes_ridge[0].plot(param_labels_ridge, -cv_results_ridge['mean_test_score'],  label='CV RMSE',    marker='o', ls='-',  color='salmon')
axes_ridge[0].set_title(f'{model_name_ridge}: Tuning Results')
axes_ridge[0].set_ylabel('RMSE (m)')
axes_ridge[0].tick_params(axis='x', rotation=45)
axes_ridge[0].legend(); axes_ridge[0].grid(True, alpha=0.3)

axes_ridge[1].scatter(y_test, y_pred_ridge, alpha=0.2, s=5, color='steelblue')
lim = [min(y_test.min(), y_pred_ridge.min()), max(y_test.max(), y_pred_ridge.max())]
axes_ridge[1].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
axes_ridge[1].set_xlabel('Actual Range (m)'); axes_ridge[1].set_ylabel('Predicted Range (m)')
axes_ridge[1].set_title(f'{model_name_ridge}: Predicted vs Actual\nRMSE={rmse_ridge:.4f} m  R²={r2_ridge:.4f}')
axes_ridge[1].legend(); axes_ridge[1].grid(alpha=0.3)

residuals_ridge = y_test.values - y_pred_ridge
axes_ridge[2].scatter(y_pred_ridge, residuals_ridge, alpha=0.2, s=5, color='darkorange')
axes_ridge[2].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes_ridge[2].set_xlabel('Predicted Range (m)'); axes_ridge[2].set_ylabel('Residual (m)')
axes_ridge[2].set_title(f'{model_name_ridge}: Residuals\nMAE={mae_ridge:.4f} m')
axes_ridge[2].grid(alpha=0.3)

plt.tight_layout(); plt.subplots_adjust(top=0.85); plt.show()
print(f"Best Params : {grid_search_ridge.best_params_}")
print(f"RMSE={rmse_ridge:.4f} m   MAE={mae_ridge:.4f} m   R²={r2_ridge:.4f}")



In [ ]:
print("Tuning and Evaluating: Gradient Boosting...")
model_name_gb = "Gradient Boosting"

gb_model  = GradientBoostingRegressor(random_state=42)
gb_params = {
    'classifier__n_estimators' : [50, 100],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth'    : [3, 5]
}
pipeline_gb    = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', gb_model)])
grid_search_gb = GridSearchCV(pipeline_gb, gb_params, cv=3,
                               scoring='neg_root_mean_squared_error',
                               n_jobs=-1, return_train_score=True)
grid_search_gb.fit(X_train, y_train)

best_model_gb = grid_search_gb.best_estimator_
y_pred_gb     = best_model_gb.predict(X_test)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
mae_gb  = mean_absolute_error(y_test, y_pred_gb)
r2_gb   = r2_score(y_test, y_pred_gb)

fig_gb, axes_gb = plt.subplots(3, 1, figsize=(8, 15))
fig_gb.suptitle(f'Comprehensive Evaluation: {model_name_gb}', fontsize=16, fontweight='bold')

cv_results_gb   = pd.DataFrame(grid_search_gb.cv_results_)
param_labels_gb = [str(p).replace('classifier__', '') for p in cv_results_gb['params']]
axes_gb[0].plot(param_labels_gb, -cv_results_gb['mean_train_score'], label='Train RMSE', marker='s', ls='--', color='skyblue')
axes_gb[0].plot(param_labels_gb, -cv_results_gb['mean_test_score'],  label='CV RMSE',    marker='o', ls='-',  color='salmon')
axes_gb[0].set_title(f'{model_name_gb}: Tuning Results')
axes_gb[0].set_ylabel('RMSE (m)')
axes_gb[0].tick_params(axis='x', rotation=45)
axes_gb[0].legend(); axes_gb[0].grid(True, alpha=0.3)

axes_gb[1].scatter(y_test, y_pred_gb, alpha=0.2, s=5, color='steelblue')
lim = [min(y_test.min(), y_pred_gb.min()), max(y_test.max(), y_pred_gb.max())]
axes_gb[1].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
axes_gb[1].set_xlabel('Actual Range (m)'); axes_gb[1].set_ylabel('Predicted Range (m)')
axes_gb[1].set_title(f'{model_name_gb}: Predicted vs Actual\nRMSE={rmse_gb:.4f} m  R²={r2_gb:.4f}')
axes_gb[1].legend(); axes_gb[1].grid(alpha=0.3)

residuals_gb = y_test.values - y_pred_gb
axes_gb[2].scatter(y_pred_gb, residuals_gb, alpha=0.2, s=5, color='darkorange')
axes_gb[2].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes_gb[2].set_xlabel('Predicted Range (m)'); axes_gb[2].set_ylabel('Residual (m)')
axes_gb[2].set_title(f'{model_name_gb}: Residuals\nMAE={mae_gb:.4f} m')
axes_gb[2].grid(alpha=0.3)

plt.tight_layout(); plt.subplots_adjust(top=0.85); plt.show()
print(f"Best Params : {grid_search_gb.best_params_}")
print(f"RMSE={rmse_gb:.4f} m   MAE={mae_gb:.4f} m   R²={r2_gb:.4f}")



SVR

In [ ]:
from sklearn.utils import resample
X_train_svr, y_train_svr = resample(X_train, y_train, n_samples=5000, random_state=42)

print("Tuning and Evaluating: SVR (RBF Kernel)...")
model_name_svr = "SVR (RBF Kernel)"


svr_model  = SVR(kernel='rbf')
svr_params = {
    'classifier__C'    : [0.1, 1.0, 10.0],
    'classifier__gamma': ['scale', 'auto']
}
pipeline_svr    = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', svr_model)])
grid_search_svr = GridSearchCV(pipeline_svr, svr_params, cv=3,
                                scoring='neg_root_mean_squared_error',
                                n_jobs=-1, return_train_score=True)
grid_search_svr.fit(X_train_svr, y_train_svr)

best_model_svr = grid_search_svr.best_estimator_
y_pred_svr     = best_model_svr.predict(X_test)
rmse_svr = np.sqrt(mean_squared_error(y_test, y_pred_svr))
mae_svr  = mean_absolute_error(y_test, y_pred_svr)
r2_svr   = r2_score(y_test, y_pred_svr)

fig_svr, axes_svr = plt.subplots(3, 1, figsize=(8, 15))
fig_svr.suptitle(f'Comprehensive Evaluation: {model_name_svr}', fontsize=16, fontweight='bold')

cv_results_svr   = pd.DataFrame(grid_search_svr.cv_results_)
param_labels_svr = [str(p).replace('classifier__', '') for p in cv_results_svr['params']]
axes_svr[0].plot(param_labels_svr, -cv_results_svr['mean_train_score'], label='Train RMSE', marker='s', ls='--', color='skyblue')
axes_svr[0].plot(param_labels_svr, -cv_results_svr['mean_test_score'],  label='CV RMSE',    marker='o', ls='-',  color='salmon')
axes_svr[0].set_title(f'{model_name_svr}: Tuning Results')
axes_svr[0].set_ylabel('RMSE (m)')
axes_svr[0].tick_params(axis='x', rotation=45)
axes_svr[0].legend(); axes_svr[0].grid(True, alpha=0.3)

axes_svr[1].scatter(y_test, y_pred_svr, alpha=0.2, s=5, color='steelblue')
lim = [min(y_test.min(), y_pred_svr.min()), max(y_test.max(), y_pred_svr.max())]
axes_svr[1].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
axes_svr[1].set_xlabel('Actual Range (m)'); axes_svr[1].set_ylabel('Predicted Range (m)')
axes_svr[1].set_title(f'{model_name_svr}: Predicted vs Actual\nRMSE={rmse_svr:.4f} m  R²={r2_svr:.4f}')
axes_svr[1].legend(); axes_svr[1].grid(alpha=0.3)

residuals_svr = y_test.values - y_pred_svr
axes_svr[2].scatter(y_pred_svr, residuals_svr, alpha=0.2, s=5, color='darkorange')
axes_svr[2].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes_svr[2].set_xlabel('Predicted Range (m)'); axes_svr[2].set_ylabel('Residual (m)')
axes_svr[2].set_title(f'{model_name_svr}: Residuals\nMAE={mae_svr:.4f} m')
axes_svr[2].grid(alpha=0.3)

plt.tight_layout(); plt.subplots_adjust(top=0.85); plt.show()
print(f"Best Params : {grid_search_svr.best_params_}")
print(f"RMSE={rmse_svr:.4f} m   MAE={mae_svr:.4f} m   R²={r2_svr:.4f}")



Summary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==========================================
# FINAL COMPARISON SUMMARY — REGRESSION
# ==========================================

compiled_models = [
    ('model_name_rf',    'best_model_rf',    'y_pred_rf'),
    ('model_name_dt',    'best_model_dt',    'y_pred_dt'),
    ('model_name_knn',   'best_model_knn',   'y_pred_knn'),
    ('model_name_ridge', 'best_model_ridge', 'y_pred_ridge'),
    ('model_name_gb',    'best_model_gb',    'y_pred_gb'),
    ('model_name_svr',   'best_model_svr',   'y_pred_svr'),
]

final_model_names  = []
final_train_rmses  = []
final_test_rmses   = []
final_test_maes    = []
final_test_r2s     = []

for name_var, model_var, pred_var in compiled_models:
    if name_var in globals() and model_var in globals() and pred_var in globals():
        m_name  = globals()[name_var]
        m_model = globals()[model_var]
        m_pred  = globals()[pred_var]

        train_rmse = np.sqrt(mean_squared_error(y_train, m_model.predict(X_train)))
        test_rmse  = np.sqrt(mean_squared_error(y_test,  m_pred))
        test_mae   = mean_absolute_error(y_test, m_pred)
        test_r2    = r2_score(y_test, m_pred)

        final_model_names.append(m_name)
        final_train_rmses.append(train_rmse)
        final_test_rmses.append(test_rmse)
        final_test_maes.append(test_mae)
        final_test_r2s.append(test_r2)

if len(final_model_names) == 0:
    print("Error: No models found. Please run at least one model cell above.")
else:
    print(f"Generating charts for: {', '.join(final_model_names)}...\n")

    x     = np.arange(len(final_model_names))
    width = 0.35

    fig, axes = plt.subplots(3, 1, figsize=(10, 18))
    fig.suptitle('Final Comparison: Best Tuned Regression Models', fontsize=15, fontweight='bold')

    # ── Chart 1: Train vs Test RMSE ───────────────────────────────────────────
    rects1 = axes[0].bar(x - width/2, final_train_rmses, width, label='Train RMSE', color='skyblue',  edgecolor='black')
    rects2 = axes[0].bar(x + width/2, final_test_rmses,  width, label='Test RMSE',  color='salmon',   edgecolor='black')
    axes[0].set_ylabel('RMSE (m)', fontweight='bold')
    axes[0].set_title('Train vs Test RMSE\n(lower is better)')
    axes[0].set_xticks(x); axes[0].set_xticklabels(final_model_names, rotation=45, ha='right')
    axes[0].legend(); axes[0].grid(axis='y', linestyle='--', alpha=0.7)
    for rect in rects1 + rects2:
        h = rect.get_height()
        axes[0].annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width()/2, h),
                         xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

    # ── Chart 2: Test MAE ─────────────────────────────────────────────────────
    colors_mae = ['#2ecc71' if v == min(final_test_maes) else '#3498db' for v in final_test_maes]
    rects3 = axes[1].bar(x, final_test_maes, width*1.5, color=colors_mae, edgecolor='black')
    axes[1].set_ylabel('MAE (m)', fontweight='bold')
    axes[1].set_title('Test MAE\n(lower is better, green = best)')
    axes[1].set_xticks(x); axes[1].set_xticklabels(final_model_names, rotation=45, ha='right')
    axes[1].grid(axis='y', linestyle='--', alpha=0.7)
    for rect in rects3:
        h = rect.get_height()
        axes[1].annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width()/2, h),
                         xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

    # ── Chart 3: Test R² ──────────────────────────────────────────────────────
    colors_r2 = ['#2ecc71' if v == max(final_test_r2s) else '#3498db' for v in final_test_r2s]
    rects4 = axes[2].bar(x, final_test_r2s, width*1.5, color=colors_r2, edgecolor='black')
    axes[2].set_ylabel('R²', fontweight='bold')
    axes[2].set_title('Test R²\n(higher is better, green = best)')
    axes[2].set_xticks(x); axes[2].set_xticklabels(final_model_names, rotation=45, ha='right')
    axes[2].set_ylim(0, 1.1)
    axes[2].grid(axis='y', linestyle='--', alpha=0.7)
    for rect in rects4:
        h = rect.get_height()
        axes[2].annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width()/2, h),
                         xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.show()

    # ── Print summary table ───────────────────────────────────────────────────
    import pandas as pd
    summary_df = pd.DataFrame({
        'Model'   : final_model_names,
        'Train RMSE (m)': final_train_rmses,
        'Test RMSE (m)' : final_test_rmses,
        'Test MAE (m)'  : final_test_maes,
        'Test R²'       : final_test_r2s,
    }).sort_values('Test RMSE (m)').reset_index(drop=True)

    print(summary_df.to_string(index=False, float_format='{:.4f}'.format))

    best = summary_df.iloc[0]
    print(f"\n✅ BEST MODEL : {best['Model']}")
    print(f"   Test RMSE  : {best['Test RMSE (m)']:.4f} m")
    print(f"   Test MAE   : {best['Test MAE (m)']:.4f} m")
    print(f"   Test R²    : {best['Test R²']:.4f}")